# Decision trees at the parcel desk

A sorting team knows each parcel's weight and packaging type at arrival. A fitted tree asks one question at a time, then sends the parcel to **Auto** or **Review**. Later we keep the inputs and predict **processing minutes**.

**Live teaching route.** First run **Runtime → Run all**. Start screen sharing at **Score the first cut**. Change the cut in the short code cell and run it again. Then open **Fit a tree and trace the parcel** and **Choose where to stop**. The last sections support the deeper explanations on feature credit and numerical leaves.

**If you joined midway.** The first toy has eight past parcels for fitting and six separate route examples for evaluation. The deeper stories use another synthetic batch so we can inspect more levels. Review is class 1. The same data and hand calculations appear in the [Decision Tree Lab](https://sharanharsoor.github.io/ml-learning-lab/decision-trees/).

In [ ]:
from collections import Counter
from html import escape
from math import isclose, log2
from statistics import mean

from IPython.display import HTML, display
from sklearn import __version__ as sklearn_version
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

train = [
    (1, 'A', 0, 2), (2, 'B', 0, 2), (3, 'B', 0, 2), (4, 'B', 1, 6),
    (5, 'B', 1, 8), (6, 'B', 1, 8), (7, 'A', 0, 4), (8, 'B', 1, 8),
]
held_out = [
    (2, 'A', 0), (3, 'B', 0), (5, 'B', 1), (8, 'B', 1),
    (6, 'A', 1), (7, 'B', 0),
]
routes = ('Auto', 'Review')
print(f'{len(train)} training parcels • {len(held_out)} held-out routes • scikit-learn {sklearn_version}')

## Score the first cut

Start with four Auto and four Review parcels. The next label has entropy **1 bit** and Gini impurity **0.5**. Entropy averages each label's surprise, with the label's probability as its weight. Gini measures the chance that two independent draws from the same node have different labels.

Splitting on weight at 3.5 sends three Auto parcels left. Five go right, including four Review. Multiply each child's impurity by its **share of the parent**, then subtract the weighted value from the parent's impurity.

The next code cell scores the existing weight and packaging questions. In the following cell, edit **threshold** or **criterion** and rerun. Try **3.5**, then **2.5** to compare one exact midpoint with a coarse candidate. The **2.5** cutoff leaves more mixed labels in the larger child.

In [ ]:
def counts_score(counts, criterion):
    total = sum(counts)
    p = [count / total for count in counts if count] if total else []
    if not p:
        return 0.0
    if criterion == 'entropy':
        return sum(-q * log2(q) for q in p)
    if criterion == 'gini':
        return 1 - sum(q ** 2 for q in p)
    raise ValueError('Choose entropy or gini.')


def split_score(children, criterion):
    parent = [sum(values) for values in zip(*children)]
    n = sum(parent)
    before = counts_score(parent, criterion)
    scores = [counts_score(counts, criterion) for counts in children]
    after = sum(sum(counts) / n * score for counts, score in zip(children, scores))
    return before, scores, after, before - after


def binary_counts(rows, goes_left):
    left = Counter(row[2] for row in rows if goes_left(row))
    right = Counter(row[2] for row in rows if not goes_left(row))
    return [[group[0], group[1]] for group in (left, right)]


fixed_candidates = {
    'Weight ≤ 3.5': binary_counts(train, lambda row: row[0] <= 3.5),
    'Packaging A': binary_counts(train, lambda row: row[1] == 'A'),
}

for question, children in fixed_candidates.items():
    before, scores, after, gain = split_score(children, 'entropy')
    print(f'{question}   weighted child entropy {after:.4f} bits   gain {gain:.4f} bits')

assert isclose(split_score(fixed_candidates['Weight ≤ 3.5'], 'entropy')[3], .5487949406953986)
assert isclose(split_score(fixed_candidates['Weight ≤ 3.5'], 'gini')[3], .3)

In [ ]:
threshold = 3.5
criterion = 'entropy'
children = binary_counts(train, lambda row: row[0] <= threshold)
if not all(map(sum, children)):
    print('Choose a threshold that leaves at least one parcel on each side.')
else:
    before, scores, after, gain = split_score(children, criterion)
    for name, counts, score in zip(('Yes', 'No'), children, scores):
        print(f'{name}   {counts[0]} Auto + {counts[1]} Review   {criterion} {score:.4f}')
    print(f'{before:.4f} parent − {after:.4f} weighted children = {gain:.4f} gain')

In [ ]:
theme = """
<style>
.dt-out{font:16px/1.48 system-ui;color:#193d38;border:1px solid #a7c6b9;border-radius:14px;
  background:#f6fbf8;padding:16px;margin:10px 0;max-width:980px}
.dt-out h3{font-size:1.35em;margin:0 0 .45em}
.dt-out h4{font-size:1.05em;margin:0 0 .45em}
.dt-out p{margin:.5em 0}
.dt-out details{border-block-start:1px solid #c7dbd1;margin-top:.7em;padding:.45em 0}
.dt-out summary{cursor:pointer;font-weight:700}
.dt-out table{border-collapse:collapse;text-align:left;max-width:100%;margin:.45em 0}
.dt-out :is(th,td){padding:.32em .6em;border-bottom:1px solid #d6e7dc}
.dt-out .scroll{overflow-x:auto}
.dt-out .band{display:flex;height:11px;border-radius:5px;overflow:hidden;margin:.18em 0 .35em}
.dt-out .branches{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,175px),1fr));gap:.6em}
.dt-out .branch{background:white;border:1px solid #c7dbd1;border-radius:8px;padding:.65em}
.dt-out svg{display:block;width:100%;min-width:560px}
.dt-out .crumb{font-size:.9em;color:#4d655d}
</style>
"""


def panel(title, content):
    display(HTML(theme + f'<section class="dt-out" data-visual="decision-trees">'
                f'<h3>{escape(title)}</h3>{content}</section>'))


def reveal(title, body, opened=False):
    flag = ' open' if opened else ''
    return f'<details{flag}><summary>{escape(title)}</summary>{body}</details>'


def split_cards(children, names, outcomes, criterion):
    before, scores, after, gain = split_score(children, criterion)
    total = sum(map(sum, children))
    palette = ('#087669', '#ae5c3b', '#64479e')
    cards = []
    for name, counts, score in zip(names, children, scores):
        n = sum(counts)
        bars = ''.join(f'<span style="width:{count / n * 100}%;background:{color}"></span>'
                       for count, color in zip(counts, palette))
        labels = ' · '.join(f'{count} {label}' for count, label in zip(counts, outcomes))
        cards.append(f'<article class="branch"><h4>{escape(name)}</h4><p>{n}/{total} of parent</p>'
                     f'<div class="band" aria-hidden="true">{bars}</div><p>{labels}</p>'
                     f'<p>{criterion.capitalize()} {score:.4f}</p></article>')
    terms = ' + '.join(f'({sum(counts)}/{total}) × {score:.4f}'
                       for counts, score in zip(children, scores))
    return (f'<div class="branches">{"".join(cards)}</div>'
            f'<p>Weighted child score = {terms} ≈ <strong>{after:.4f}</strong></p>'
            f'<p>Gain = {before:.4f} − {after:.4f} ≈ <strong>{gain:.4f}</strong></p>')


question_panels = []
for name, children in fixed_candidates.items():
    for criterion in ('entropy', 'gini'):
        question_panels.append(reveal(
            f'{name} · {criterion.capitalize()}',
            split_cards(children, ('Yes child', 'No child'), routes, criterion),
            opened=(name == 'Weight ≤ 3.5' and criterion == 'entropy'),
        ))
panel('Compare the first questions', ''.join(question_panels))

### Which numeric cutoffs get tested?

Sort the distinct weights. Any cutoff between two successive observed values gives the same training partition, so exact search can test a midpoint per gap. Histogram approaches use a smaller set of admissible edges and can miss the exact winner. The categorical question still competes for the split.

Packaging types and output classes are separate ideas. A, B, and C can be three input categories. Auto, Review, and Hold can be three outcomes. A binary tree can group the input categories into two children while predicting any of the three outcomes. A multiway method can create one branch per category. The same weighted average extends to every child it creates.

In [ ]:
values = sorted({row[0] for row in train})
midpoints = [(a + b) / 2 for a, b in zip(values, values[1:])]
coarse_edges = [2.5, 4.5, 6.5]

for mode, thresholds in [('Exact midpoints', midpoints), ('Coarse edges', coarse_edges)]:
    scored = [
        (cut, split_score(binary_counts(train, lambda row: row[0] <= cut), 'entropy')[3])
        for cut in thresholds
    ]
    best_cut, best_gain = max(scored, key=lambda item: item[1])
    print(f'{mode}   best numeric cutoff {best_cut:g}   gain {best_gain:.4f} bits')

a, b, c = [3, 1, 0], [0, 3, 1], [0, 0, 4]
outcomes = ('Auto', 'Review', 'Hold')
options = [
    ('Packaging A or other', [a, [x + y for x, y in zip(b, c)]],
     ('Packaging A', 'Packaging B or C')),
    ('One branch per packaging type', [a, b, c], ('Packaging A', 'Packaging B', 'Packaging C')),
]
panel('Two or three children', ''.join(
    reveal(name, split_cards(groups, children, outcomes, 'entropy'), opened=i == 0)
    for i, (name, groups, children) in enumerate(options)
))

## Fit a tree and trace the parcel

The eight **training** parcels teach a root question. Each mixed child gets its own candidate search. The notebook fits scikit-learn using a 0/1 column for packaging B, a one-hot indicator for this two-category feature. The new parcel at **weight 7, packaging A** follows the heavier branch and then uses packaging to reach Auto.

At any decision node, the denominator for the split's weighted children is the number of rows **at that parent**. At a leaf, any displayed class share is based on the data that reached it.

In [ ]:
def input_columns(rows):
    return [[row[0], int(row[1] == 'B')] for row in rows]


def fit_classifier(depth, min_leaf=1, alpha=0):
    return DecisionTreeClassifier(
        criterion='entropy', max_depth=depth, min_samples_leaf=min_leaf,
        ccp_alpha=alpha, random_state=0,
    ).fit(input_columns(train), [row[2] for row in train])


classifier = fit_classifier(2)
parcel = [7, 0]
assert classifier.predict([parcel]).tolist() == [0]

In [ ]:
def tree_picture(model, features, example, numerical=False):
    tree = model.tree_
    on_path = set(model.decision_path([example]).indices)
    xy = {}
    edges = []
    split_notes = []
    next_slot = 0
    unit = ' min' if numerical else ''

    def position(node, depth):
        nonlocal next_slot
        left, right = tree.children_left[node], tree.children_right[node]
        y = 44 + depth * 112
        count = tree.n_node_samples[node]
        if left < 0:
            x = 72 + 147 * next_slot
            next_slot += 1
            value = tree.value[node][0]
            first = f'{value[0]:g}{unit}' if numerical else routes[model.classes_[value.argmax()]]
            xy[node] = (x, y, first, f'{count} training rows')
            return x

        x = (position(left, depth + 1) + position(right, depth + 1)) / 2
        feature = features[tree.feature[node]]
        threshold = tree.threshold[node]
        question = 'Packaging A?' if feature == 'Packaging B' and isclose(threshold, .5) else f'{feature} ≤ {threshold:g}?'
        xy[node] = (x, y, question, f'{count} training rows')
        for side, child in [('Yes', left), ('No', right)]:
            cx, cy, *_ = xy[child]
            highlighted = child in on_path
            color = '#64479e' if highlighted else '#94b2a7'
            edges.append(f'<path d="M{x} {y+24} L{cx} {cy-24}" stroke="{color}" '
                         f'stroke-width="{4 if highlighted else 2}" />'
                         f'<text x="{(x+cx)/2}" y="{(y+cy)/2-5}" '
                         f'font-size="12" text-anchor="middle" '
                         f'paint-order="stroke" stroke="#f6fbf8" stroke-width="4">{side}</text>')
        wl, wr = tree.weighted_n_node_samples[[left, right]]
        total = wl + wr
        il, ir = tree.impurity[[left, right]]
        weighted = wl / total * il + wr / total * ir
        gain = tree.impurity[node] - weighted
        split_notes.append((depth, node, reveal(
            f'Depth {depth} · {question}',
            f'<p>Parent {total:g} rows · child scores {il:.4f} and {ir:.4f}</p>'
            f'<p>Weighted child score = ({wl:g}/{total:g}) × {il:.4f} + '
            f'({wr:g}/{total:g}) × {ir:.4f} ≈ <strong>{weighted:.4f}</strong>.</p>'
            f'<p>Gain = {tree.impurity[node]:.4f} − {weighted:.4f} ≈ {gain:.4f}. '
            f'Across all {tree.n_node_samples[0]} training rows, '
            f'credit = ({total:g}/{tree.n_node_samples[0]}) × {gain:.4f} '
            f'≈ {total / tree.n_node_samples[0] * gain:.4f}.</p>',
        )))
        return x

    position(0, 0)
    nodes = []
    for node, (x, y, first, second) in xy.items():
        active = node in on_path
        fill = '#eee6fc' if active else 'white'
        border = '#64479e' if active else '#a4c2b5'
        nodes.append(f'<g><rect x="{x-63}" y="{y-24}" width="126" height="48" fill="{fill}" '
                     f'stroke="{border}" stroke-width="{3 if active else 1}" rx="8" />'
                     f'<text x="{x}" y="{y-3}" text-anchor="middle" font-size="12" font-weight="700">{escape(first)}</text>'
                     f'<text x="{x}" y="{y+13}" text-anchor="middle" font-size="11">{escape(second)}</text></g>')
    height = 90 + tree.max_depth * 112
    width = max(294, next_slot * 147)
    diagram = (f'<div class="scroll"><svg viewBox="0 0 {width} {height}" '
               f'role="img" aria-label="Fitted tree. The purple path traces the input parcel.">'
               f'{"".join(edges)}{"".join(nodes)}</svg></div>')
    result = model.predict([example])[0]
    prediction = f'{result:g} minutes' if numerical else routes[result]
    return (f'<p><strong>Prediction {prediction}</strong> · purple shows the path for this parcel.</p>'
            + diagram + ''.join(note for _, _, note in sorted(split_notes)))


panel('Trace weight 7 and packaging A', tree_picture(classifier, ('Weight', 'Packaging B'), parcel))

### Read the immediate parent further down

This example uses a separate **16-row** parcel batch so we can see five levels of splits. Trace weight 7 and open **Depth 3 · Weight ≤ 5.5?**. Its parent has seven rows. One child has three rows and zero entropy. The other has four rows and one bit. The weighted score is 4/7 ≈ **0.5714 bits**. The earlier questions explain which records arrived. The root count of 16 enters only if we assign this split tree-wide credit.

In [ ]:
route_batch = [0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1]
route_tree = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=0).fit(
    [[weight] for weight in range(1, 17)], route_batch,
)
panel('Which rows count at depth 3?', tree_picture(route_tree, ('Weight',), [7]))
assert route_tree.get_depth() == 5

## Choose where to stop

Compare the training scores with the six fixed evaluation parcels. Depth 1 asks the weight question. Depth 2 can ask about packaging too. That second question makes the training leaves pure, but one evaluation parcel ends up on the wrong side compared with the simpler tree.

If these evaluation rows guide the selection of depth, they serve as validation. An independent test set would estimate the selected model's performance. A minimum leaf size acts while the model grows. Cost-complexity pruning compares weighted leaf impurity plus a per-leaf penalty after fitting a larger tree.

In [ ]:
def depth_case(depth, leaf_size=1):
    model = fit_classifier(depth, leaf_size)
    train_score = model.score(input_columns(train), [row[2] for row in train])
    y = [row[2] for row in held_out]
    predictions = model.predict(input_columns(held_out))
    matrix = confusion_matrix(y, predictions, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    table = ('<table><caption>Actual rows, predicted columns</caption>'
             '<thead><tr><th></th><th>Auto</th><th>Review</th></tr></thead>'
             f'<tbody><tr><th>Auto</th><td>TN {tn}</td><td>FP {fp}</td></tr>'
             f'<tr><th>Review</th><td>FN {fn}</td><td>TP {tp}</td></tr></tbody></table>')
    return (f'<p>Training {train_score:.1%} · held-out {(tn + tp) / len(y):.1%}</p>' + table)


panel('One question or two?', reveal('Depth 1 · one question', depth_case(1), opened=True)
      + reveal('Depth 2 · up to two questions', depth_case(2))
      + reveal('Depth 2 · minimum two rows per leaf', depth_case(2, 2)))
assert confusion_matrix([row[2] for row in held_out],
                        classifier.predict(input_columns(held_out)), labels=[0, 1]).tolist() == [[2, 1], [1, 2]]
print('Leaf counts by pruning penalty', {a: int(fit_classifier(2, alpha=a).get_n_leaves()) for a in (.2, .47)})

## Give the deeper tree's features credit

A new batch of **20** training parcels adds scan score. To score the split itself, weight its children by their share of the immediate parent. To assign tree-wide credit, multiply the resulting gain by the share of the **root's twenty** rows that reaches the node. Add credits from every split using the feature. A feature the tree never uses gets zero.

The first three weight cuts stay the same when depth increases from 4 to 5. The new scan score split adds more total credit, so weight's percentage can go down while its raw credit stays put.

In [ ]:
packaging = 'AABAABBBABAAABBAAAAA'
scan = '33242422224244223423'
deep_labels = [int(value) for value in '11100001011011101111']
deep_inputs = [[i + 1, int(pkg == 'B'), int(score)]
               for i, (pkg, score) in enumerate(zip(packaging, scan))]
deep_features = ('Weight', 'Packaging B', 'Scan score')
deep_classifiers = {
    depth: DecisionTreeClassifier(criterion='entropy', max_depth=depth, random_state=0)
    .fit(deep_inputs, deep_labels) for depth in (4, 5)
}
reference_parcel = [11.5, 0, 4]


def credit_table(model):
    tree = model.tree_
    credits = [0.0] * len(deep_features)
    root = tree.weighted_n_node_samples[0]
    for node, feature in enumerate(tree.feature):
        if feature < 0:
            continue
        left, right = tree.children_left[node], tree.children_right[node]
        counts = tree.weighted_n_node_samples
        impurity = tree.impurity
        credit = (counts[node] * impurity[node] - counts[left] * impurity[left]
                  - counts[right] * impurity[right]) / root
        credits[feature] += credit
    rows = ''.join(f'<tr><th>{feature}</th><td>{credit:.4f}</td><td>{share:.2%}</td></tr>'
                   for feature, credit, share in zip(deep_features, credits, model.feature_importances_))
    assert all(isclose(credit / sum(credits), share, abs_tol=1e-10)
               for credit, share in zip(credits, model.feature_importances_))
    return ('<table><thead><tr><th>Feature</th><th>Raw credit in bits</th>'
            f'<th>Share of total</th></tr></thead><tbody>{rows}</tbody></table>')


panel('Which splits earn credit?', ''.join(
    reveal(f'Maximum depth {depth}', credit_table(model) +
           tree_picture(model, deep_features, reference_parcel), opened=depth == 4)
    for depth, model in deep_classifiers.items()
))

## Predict processing minutes

Keep the inputs and change the target from route to observed time. For squared error, each leaf predicts its mean. Its MSE is the average of squared distances from that mean. For the same 3.5 weight cut in the small batch, the two means are 2 and 6.8 minutes. The weighted child MSE is 1.6 minutes squared, down from 7 at the root.

The deeper batch records a duration for each fitting parcel. Follow the same parcel through a depth-4 and depth-5 regression tree. At maximum depth 4 it predicts 13.8 minutes. At depth 5 the scan score distinguishes a 13-minute leaf from a 15-minute leaf. The regression prediction always comes from the leaf reached. A model's error on unseen parcels needs new parcels with measured times.

In [ ]:
times = [row[3] for row in train]
small_regressor = DecisionTreeRegressor(max_depth=1, random_state=0).fit(input_columns(train), times)
low_mean, high_mean = small_regressor.predict([[1, 0], [5, 1]])
left = [row[3] for row in train if row[0] <= 3.5]
right = [row[3] for row in train if row[0] > 3.5]


def mse_at_mean(group):
    center = mean(group)
    return mean((target - center) ** 2 for target in group)


weighted_mse = (len(left) * mse_at_mean(left) + len(right) * mse_at_mean(right)) / len(train)
assert isclose(weighted_mse, 1.6)
assert isclose(low_mean, 2) and isclose(high_mean, 6.8)
print(f'Child means {low_mean:g} and {high_mean:g} min   weighted MSE {weighted_mse:g} min²')
print(f'Root MSE {mse_at_mean(times):g} min²   reduction {mse_at_mean(times) - weighted_mse:g} min²')

deep_times = [19, 19, 19, 10, 10, 10, 10, 19, 12, 19, 14.5, 13, 15.5, 19, 19, 14, 9, 9, 9, 9]
regressors = {
    depth: DecisionTreeRegressor(max_depth=depth, random_state=0).fit(deep_inputs, deep_times)
    for depth in (4, 5)
}
cases = [
    ('Depth 4 · scan score 4', regressors[4], [11.5, 0, 4]),
    ('Depth 5 · scan score 4', regressors[5], [11.5, 0, 4]),
    ('Depth 5 · scan score 2', regressors[5], [11.5, 0, 2]),
]
assert all(isclose(model.predict([row])[0], expected)
           for (_, model, row), expected in zip(cases, (13.8, 15, 13)))
panel('Compare the leaf means for a new parcel', ''.join(
    reveal(name, tree_picture(model, deep_features, row, numerical=True), opened=i == 0)
    for i, (name, model, row) in enumerate(cases)
))

## Connect the results

The candidate search asks which question improves the current node. The validation check asks whether more questions help on fresh records. Feature credit adds up the training improvements across the full tree. Changing the target to minutes changes the node score and leaf value while keeping that workflow intact.

Return to the [Decision Tree Lab](https://sharanharsoor.github.io/ml-learning-lab/decision-trees/#python) for the same first calculation in the browser and a guided replay of each visual.